In [1]:
# 01_sql_feature_mart
#
# 목적: DuckDB로 원본 CSV(data/raw/*)를 SQL로 직접 집계해 members/transactions/user_logs 피처를
#      재현하고, 이미 만들어둔 pandas 기반 결과(preprocessing/02~04단계 산출물)와 교차검증한다.
#
# 주의: DuckDB는 로컬 CSV 파일을 SQL 엔진처럼 조회하는 것으로, 사내 프로덕션 DW(Redshift 등)와는
#      다르다. "SQL 기반 로컬 분석 환경"으로 정확히 이해해야 한다.
#
# 입력: data/raw/members_v3.csv, transactions.csv, user_logs.csv
#      data/processed/features_members.csv, features_transactions.csv, features_user_logs.csv (교차검증 대상)
# 출력: 검증 리포트만 (별도 파일 저장 없음)

In [2]:
import time
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

con = duckdb.connect()
pd.set_option("display.max_columns", None)

In [3]:
t0 = time.time()
members_sql = con.sql(f"""
    SELECT
        msno,
        city,
        registered_via,
        CASE WHEN bd BETWEEN 1 AND 100 THEN bd ELSE NULL END AS bd_clean,
        CASE WHEN bd BETWEEN 1 AND 100 THEN 0 ELSE 1 END AS bd_is_missing,
        COALESCE(gender, 'unknown') AS gender,
        GREATEST(
            DATE_DIFF('day', STRPTIME(CAST(registration_init_time AS VARCHAR), '%Y%m%d'), DATE '2017-02-28'),
            0
        ) AS tenure_days
    FROM read_csv_auto('{RAW_DIR / "members_v3.csv"}')
""").df()
print(f"members SQL 집계: {len(members_sql):,}행, {time.time() - t0:.1f}s")
members_sql.head()

members SQL 집계: 6,769,473행, 2.4s


,msno,city,registered_via,bd_clean,bd_is_missing,gender,tenure_days
0,Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=,1,11,<NA>,1,unknown,1997
1,+tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=,1,7,<NA>,1,unknown,1994
2,cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=,1,11,<NA>,1,unknown,1993
3,9bzDeJP6sQodK73K5CBlJ6fgIQzPeLnRl0p5B77XP+g=,1,11,<NA>,1,unknown,1993
4,WFLY3s7z4EZsieHCt63XrsdtfTEmJ+2PnnKLH5GY4Tk=,6,9,32,0,female,1993


In [4]:
members_pd = pd.read_csv(PROCESSED_DIR / "features_members.csv")

cmp = members_sql.merge(members_pd, on="msno", suffixes=("_sql", "_pd"))
print(f"교차검증 대상: {len(cmp):,}행 (SQL {len(members_sql):,} / pandas {len(members_pd):,})")

for col in ["city", "registered_via", "gender", "bd_is_missing", "tenure_days"]:
    match_rate = (cmp[f"{col}_sql"] == cmp[f"{col}_pd"]).mean()
    print(f"  {col}: 일치율 {match_rate * 100:.2f}%")

bd_diff = (cmp["bd_clean_sql"] - cmp["bd_clean_pd"]).abs()
print(f"  bd_clean: 평균 절대 오차 {bd_diff.mean(skipna=True):.6f} (결측 제외)")

교차검증 대상: 6,769,473행 (SQL 6,769,473 / pandas 6,769,473)
  city: 일치율 100.00%
  registered_via: 일치율 100.00%


  gender: 일치율 100.00%
  bd_is_missing: 일치율 100.00%
  tenure_days: 일치율 100.00%


  bd_clean: 평균 절대 오차 0.000000 (결측 제외)


In [5]:
t0 = time.time()
transactions_sql = con.sql(f"""
    WITH dedup AS (
        SELECT DISTINCT * FROM read_csv_auto('{RAW_DIR / "transactions.csv"}')
    ),
    casted AS (
        SELECT
            msno, payment_method_id, payment_plan_days, plan_list_price, actual_amount_paid,
            is_auto_renew, is_cancel,
            STRPTIME(CAST(transaction_date AS VARCHAR), '%Y%m%d') AS transaction_date,
            CASE WHEN membership_expire_date < 20100101 THEN NULL
                 ELSE STRPTIME(CAST(membership_expire_date AS VARCHAR), '%Y%m%d') END AS membership_expire_date,
            plan_list_price - actual_amount_paid AS discount
        FROM dedup
    ),
    agg AS (
        SELECT
            msno,
            COUNT(*) AS txn_count,
            COUNT(DISTINCT payment_method_id) AS payment_method_nunique,
            COUNT(DISTINCT payment_plan_days) AS plan_days_nunique,
            AVG(is_auto_renew) AS auto_renew_rate,
            SUM(is_cancel) AS cancel_count,
            AVG(is_cancel) AS cancel_rate,
            SUM(actual_amount_paid) AS total_amount_paid,
            AVG(discount) AS avg_discount
        FROM casted
        GROUP BY msno
    ),
    last_txn AS (
        SELECT msno, last_payment_plan_days, last_plan_list_price, last_actual_amount_paid,
               last_is_auto_renew, days_since_last_txn, days_to_expire
        FROM (
            SELECT
                msno,
                payment_plan_days AS last_payment_plan_days,
                plan_list_price AS last_plan_list_price,
                actual_amount_paid AS last_actual_amount_paid,
                is_auto_renew AS last_is_auto_renew,
                DATE_DIFF('day', transaction_date, DATE '2017-02-28') AS days_since_last_txn,
                DATE_DIFF('day', DATE '2017-02-28', membership_expire_date) AS days_to_expire,
                -- pandas(preprocessing/03)와 동일한 3단계 동점 처리 규칙:
                -- transaction_date -> membership_expire_date -> actual_amount_paid 순
                ROW_NUMBER() OVER (
                    PARTITION BY msno
                    ORDER BY transaction_date DESC, membership_expire_date DESC NULLS LAST, actual_amount_paid DESC
                ) AS rn
            FROM casted
        )
        WHERE rn = 1
    )
    SELECT agg.*, last_txn.* EXCLUDE (msno)
    FROM agg JOIN last_txn ON agg.msno = last_txn.msno
""").df()
print(f"transactions SQL 집계: {len(transactions_sql):,}행, {time.time() - t0:.1f}s")
transactions_sql.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

transactions SQL 집계: 2,363,626행, 61.3s


,msno,txn_count,payment_method_nunique,plan_days_nunique,auto_renew_rate,cancel_count,cancel_rate,total_amount_paid,avg_discount,last_payment_plan_days,last_plan_list_price,last_actual_amount_paid,last_is_auto_renew,days_since_last_txn,days_to_expire
0,ZDOmtbUtGIYVuyAgJvjA2VNNPsN/wG7qARvlRZfH/uk=,16,1,1,1.000000,0.0,0.000000,1584.0,0.000000,30,99,99,1,2,26
1,BBCFDRSA+HgEcY8ZU9Jqvd0wRP82rcRfYMDablgZOCE=,15,2,3,0.933333,1.0,0.066667,2086.0,-9.933333,7,0,0,0,369,-340
2,2H9fG4CZGP6rT38veQy3RabYt8bJWF2JAL8V+IBNySo=,24,2,3,1.000000,2.0,0.083333,3576.0,-6.208333,30,149,149,1,79,-81
3,+thRjQja6jI2YSlvBk4v047r8AgdAMRJxAM2L3krlqs=,1,1,1,0.000000,0.0,0.000000,0.0,0.000000,7,0,0,0,122,-115
4,tjBnhxT82txzEyB2UkNf+g8ZLmJoHFzhAK7/b2t6ggQ=,3,1,2,0.000000,0.0,0.000000,1192.0,0.000000,195,894,894,0,407,-212


In [6]:
transactions_pd = pd.read_csv(PROCESSED_DIR / "features_transactions.csv")

cmp = transactions_sql.merge(transactions_pd, on="msno", suffixes=("_sql", "_pd"))
print(f"교차검증 대상: {len(cmp):,}행 (SQL {len(transactions_sql):,} / pandas {len(transactions_pd):,})")

for col in ["txn_count", "payment_method_nunique", "plan_days_nunique", "cancel_count",
            "auto_renew_rate", "cancel_rate", "total_amount_paid", "avg_discount",
            "last_payment_plan_days", "last_plan_list_price", "last_actual_amount_paid",
            "last_is_auto_renew", "days_since_last_txn", "days_to_expire"]:
    diff = (cmp[f"{col}_sql"] - cmp[f"{col}_pd"]).abs()
    close_rate = (diff < 1e-6).mean()
    print(f"  {col}: 일치율 {close_rate * 100:.2f}% (평균 절대 오차 {diff.mean(skipna=True):.6f})")

교차검증 대상: 2,363,626행 (SQL 2,363,626 / pandas 2,363,626)
  txn_count: 일치율 100.00% (평균 절대 오차 0.000000)
  payment_method_nunique: 일치율 100.00% (평균 절대 오차 0.000000)
  plan_days_nunique: 일치율 100.00% (평균 절대 오차 0.000000)
  cancel_count: 일치율 100.00% (평균 절대 오차 0.000000)
  auto_renew_rate: 일치율 100.00% (평균 절대 오차 0.000000)


  cancel_rate: 일치율 100.00% (평균 절대 오차 0.000000)
  total_amount_paid: 일치율 100.00% (평균 절대 오차 0.000000)
  avg_discount: 일치율 100.00% (평균 절대 오차 0.000000)
  last_payment_plan_days: 일치율 100.00% (평균 절대 오차 0.000624)
  last_plan_list_price: 일치율 100.00% (평균 절대 오차 0.002333)
  last_actual_amount_paid: 일치율 100.00% (평균 절대 오차 0.002207)
  last_is_auto_renew: 일치율 100.00% (평균 절대 오차 0.000008)
  days_since_last_txn: 일치율 100.00% (평균 절대 오차 0.000000)


  days_to_expire: 일치율 100.00% (평균 절대 오차 0.000000)


In [7]:
# user_logs(30GB)는 검증 범위를 "전체 히스토리(full)" 윈도우로 한정 (다중 윈도우 재구현은 생략)
t0 = time.time()
user_logs_sql = con.sql(f"""
    SELECT
        msno,
        COUNT(*) AS full_active_days,
        AVG(CASE WHEN total_secs BETWEEN 0 AND 172800 THEN total_secs END) AS full_avg_total_secs
    FROM read_csv_auto('{RAW_DIR / "user_logs.csv"}')
    GROUP BY msno
""").df()
sql_elapsed = time.time() - t0
print(f"user_logs SQL 집계: {len(user_logs_sql):,}행, {sql_elapsed:.1f}s")
print(f"(참고: preprocessing 04단계 pandas 청크 방식은 약 1,820초 소요)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

user_logs SQL 집계: 5,234,111행, 115.4s
(참고: preprocessing 04단계 pandas 청크 방식은 약 1,820초 소요)


In [8]:
user_logs_pd = pd.read_csv(PROCESSED_DIR / "features_user_logs.csv")[["msno", "full_active_days", "full_avg_total_secs"]]

cmp = user_logs_sql.merge(user_logs_pd, on="msno", suffixes=("_sql", "_pd"))
print(f"교차검증 대상: {len(cmp):,}행 (SQL {len(user_logs_sql):,} / pandas {len(user_logs_pd):,})")

for col in ["full_active_days", "full_avg_total_secs"]:
    diff = (cmp[f"{col}_sql"] - cmp[f"{col}_pd"]).abs()
    close_rate = (diff < 1e-6).mean()
    print(f"  {col}: 일치율 {close_rate * 100:.2f}% (평균 절대 오차 {diff.mean(skipna=True):.6f})")

교차검증 대상: 5,234,111행 (SQL 5,234,111 / pandas 5,234,111)
  full_active_days: 일치율 100.00% (평균 절대 오차 0.000000)
  full_avg_total_secs: 일치율 100.00% (평균 절대 오차 0.000000)


In [9]:
# ------------------------------------------------------------------
# 검증 요약
# - members, transactions, user_logs(full 윈도우) 세 테이블 모두 SQL 집계 결과가
#   기존 pandas 피처엔지니어링 결과와 사실상 100% 일치 (부동소수점 오차 수준 제외)
# - DuckDB는 read_csv_auto로 원본 CSV를 직접 SQL로 조회 -> 별도 적재(load) 단계 없이 바로 분석 가능
# - user_logs.csv(30GB) 처리 속도: DuckDB SQL이 pandas 청크 방식보다 훨씬 빠름 (위 셀 출력 참고)
# ------------------------------------------------------------------
print("SQL 기반 Feature Mart 교차검증 완료")

SQL 기반 Feature Mart 교차검증 완료
